# 📓 Notebook 04 — Agent with Tools

## เรียนรู้:
- ReAct Pattern: Agent คิด → เรียก Tool → ตอบ
- Tools ที่เรียก MT5+SMC โดยตรง (ไม่ผ่าน HTTP)

In [ ]:
import os, pandas as pd, numpy as np
import smartmoneyconcepts as smc
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
print(f'✅ Model: {OPENAI_MODEL}')

In [ ]:
# Data helpers
def _get_mt5(symbol='XAUUSD', tf='15m', count=500):
    try:
        import MetaTrader5 as mt5
        if not mt5.initialize(): return None
        tf_map = {'1m':mt5.TIMEFRAME_M1,'5m':mt5.TIMEFRAME_M5,'15m':mt5.TIMEFRAME_M15,
                  '1h':mt5.TIMEFRAME_H1,'4h':mt5.TIMEFRAME_H4,'1d':mt5.TIMEFRAME_D1,'1w':mt5.TIMEFRAME_W1}
        rates = mt5.copy_rates_from_pos(symbol, tf_map.get(tf,mt5.TIMEFRAME_M15), 0, count)
        mt5.shutdown()
        if rates is None: return None
        df = pd.DataFrame(rates); df['date']=pd.to_datetime(df['time'],unit='s')
        df=df.rename(columns={'tick_volume':'volume'})
        return df[['date','open','high','low','close','volume']].copy()
    except: return None

def _get_data(tf='15m', count=200):
    df = _get_mt5(tf=tf, count=count)
    if df is not None: return df
    if os.path.exists('sample_xauusd.csv'):
        return pd.read_csv('sample_xauusd.csv',parse_dates=['date']).tail(count).reset_index(drop=True)
    np.random.seed(42); base=2650; prices=base+np.cumsum(np.random.randn(count)*2.5)
    data=[{'date':pd.Timestamp('2025-01-01')+pd.Timedelta(minutes=15*i),'open':round(p+np.random.randn()*1.5,2),
           'high':round(max(p+np.random.randn()*1.5,p)+abs(np.random.randn())*3,2),
           'low':round(min(p+np.random.randn()*1.5,p)-abs(np.random.randn())*3,2),
           'close':round(p,2),'volume':int(abs(np.random.randn())*1000+500)} for i,p in enumerate(prices)]
    return pd.DataFrame(data)

def _compute(df):
    sl=20 if len(df)>=200 else 15 if len(df)>=100 else 10 if len(df)>=50 else max(5,len(df)//6)
    sw=smc.smc.swing_highs_lows(df,swing_length=sl)
    return {'swing':sw,'fvg':smc.smc.fvg(df),'bos_choch':smc.smc.bos_choch(df,sw,close_break=True),
            'ob':smc.smc.ob(df,sw),'liquidity':smc.smc.liquidity(df,sw,range_percent=0.02)}

def _summarize(df,ind,tf='15m'):
    last=df.iloc[-1]; f=ind['fvg']; b=ind['bos_choch']; o=ind['ob']; l=ind['liquidity']
    lines=[f'=== SMC {tf} (Price: {last["close"]:.2f}) ===']
    lines.append(f'FVG: Bull={((f["FVG"]==1)&f["MitigatedIndex"].isna()).sum()}, Bear={((f["FVG"]==-1)&f["MitigatedIndex"].isna()).sum()}')
    bm=b['BOS'].notna()
    if bm.any(): lb=b[bm].iloc[-1]; lines.append(f'BOS: {"Bull" if lb["BOS"]==1 else "Bear"} @ {lb["Level"]:.2f}')
    lines.append(f'OB: Bull={((o["OB"]==1)&o["MitigatedIndex"].isna()).sum()}, Bear={((o["OB"]==-1)&o["MitigatedIndex"].isna()).sum()}')
    lines.append(f'Liq: BSL={((l["Liquidity"]==1)&l["Swept"].isna()).sum()}, SSL={((l["Liquidity"]==-1)&l["Swept"].isna()).sum()}')
    return '\n'.join(lines)

print('✅ Helpers ready')

## 🔧 Cell 3: สร้าง Tools

In [ ]:
@tool
def get_smc_analysis(timeframe: str = '15m', candle_count: int = 200) -> str:
    """ดึง XAUUSD + คำนวณ SMC (FVG, BOS, OB, Liquidity)
    Args: timeframe: '1m'-'1w' | candle_count: จำนวนแท่งเทียน"""
    try:
        df=_get_data(tf=timeframe,count=candle_count); ind=_compute(df)
        return _summarize(df,ind,timeframe)
    except Exception as e: return f'Error: {e}'

@tool
def get_multi_timeframe() -> str:
    """วิเคราะห์ 3 TF (15m,1h,4h) ดู confluence"""
    r=[]
    for tf in ['15m','1h','4h']:
        try:
            df=_get_data(tf=tf,count=200); ind=_compute(df)
            r.append(f'\n--- {tf} ---\n{_summarize(df,ind,tf)}')
        except Exception as e: r.append(f'\n--- {tf} Error: {e}')
    return '\n'.join(r)

@tool
def get_stats() -> str:
    """สถิติ XAUUSD"""
    df=_get_data(count=500)
    return f'Price:{df["close"].iloc[-1]:.2f} High:{df["high"].max():.2f} Low:{df["low"].min():.2f}'

tools = [get_smc_analysis, get_multi_timeframe, get_stats]
print(f'✅ {len(tools)} tools ready')

# ทดสอบ
print(get_smc_analysis.invoke({'timeframe':'15m','candle_count':100}))

## 🤖 Cell 4: สร้าง Agent + รัน

In [ ]:
llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0.1)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'คุณคือ AI SMC/ICT Analyst วิเคราะห์ XAUUSD\n'
     'ต้องมี confluence 2-3 signals | R:R>=1:2 | ไม่ชัด→HOLD | ตอบไทย'),
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True,
    max_iterations=5, handle_parsing_errors=True,
    return_intermediate_steps=True,
)

print('✅ Agent ready!')
result = executor.invoke({'input':'วิเคราะห์ XAUUSD 15m แนะนำ BUY/SELL/HOLD','chat_history':[]})
print('\n📥 คำตอบ:', result['output'])

In [ ]:
# ดู intermediate steps
for i,(a,o) in enumerate(result.get('intermediate_steps',[]),1):
    print(f'Step {i}: {a.tool}({a.tool_input}) → {str(o)[:150]}...')

## ✅ สรุป
1. ✅ Tools เรียก MT5+SMC โดยตรง
2. ✅ ReAct pattern ทำงานจริง
3. ✅ ดู reasoning ผ่าน intermediate_steps

---
**➡️**: `05_full_trading_agent.ipynb`